# MetaJudge: Confidence Calibration (Family A)

**Task:** `metajudge_calibration` | **Version:** v4.1
**Axis:** Monitoring (Nelson & Narens 1990)
**Items:** 105 clean calibration items
**Score:** Anchor-normalized mean 1-Brier → float in [0, 1]

In [ ]:
import sys, os, json
sys.path.insert(0, "/kaggle/input/metajudge-package-v3")
DATA_ROOT = "/kaggle/input/metajudge-data-v3"

import kaggle_benchmarks as kbench
from kaggle_benchmarks import chats
from metajudge.scoring.grading_v2 import grade_item, load_registry

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Cell 2 — Scoring Formulas & Anchor Normalization
import numpy as np

def brier_item_score(is_correct: bool, confidence: float) -> float:
    y = 1.0 if is_correct else 0.0
    return 1.0 - (confidence - y) ** 2

ANCHOR_A_FLOOR = 0.75   # 1-Brier at 50% confidence (random baseline)
ANCHOR_A_CEIL  = 1.00   # Perfect calibration

def normalize(score, floor, ceil):
    return max(0.0, min(1.0, (score - floor) / (ceil - floor)))

print(f"Scoring: brier_item_score (inline), anchor A=[{ANCHOR_A_FLOOR}, {ANCHOR_A_CEIL}]")

In [ ]:
from dataclasses import dataclass

@dataclass
class CalibrationResponse:
    """Structured response for calibration items."""
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        for f in self.__dataclass_fields__.values():
            if not hasattr(self, f.name):
                setattr(self, f.name, f.default)


In [ ]:
import pandas as pd

with open(os.path.join(DATA_ROOT, "metajudge_benchmark_v1.json")) as f:
    all_cal_items = json.load(f)

with open(os.path.join(DATA_ROOT, "clean_subset_manifest.json")) as f:
    manifest = json.load(f)

cal_excluded = set(manifest["calibration"]["excluded_items"])
cal_items = [it for it in all_cal_items if it["item_id"] not in cal_excluded]
cal_df = pd.DataFrame(cal_items)

REGISTRY = load_registry(os.path.join(DATA_ROOT, "adjudication_registry.json"))


In [ ]:
# Cell 5 — Task: Confidence Calibration (Axis I — Monitoring)

@kbench.task(name="metacog_calibration", store_task=False)
def metacog_calibration(llm, item_id: str, question: str,
                        gold_answer: str) -> dict:
    """Evaluate a single calibration item.

    Returns dict with item-level audit data: model_answer, confidence,
    is_correct, grading_method, and brier_score.
    """
    with chats.new():
        prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Confidence Calibration\n"
            f"Question:\n{question}\n\n"
            "Instructions:\n"
            "1. Put only your final answer in the `answer` field.\n"
            "2. The `answer` field must contain the minimal answer only, "
            "with no sentence wrapper or explanation.\n"
            "3. If the question requests a number only, return only the number.\n"
            "4. If the question requests one word only, return only that word.\n"
            "5. Provide a confidence score from 0.0 to 1.0.\n"
            "6. Explain why you are or are not certain in `reason_for_uncertainty`.\n"
            "7. Say whether you would verify this if possible.\n\n"
            "Return valid structured output with keys: "
            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
        )
        response = llm.prompt(prompt, schema=CalibrationResponse)

    conf = max(0.0, min(1.0, response.confidence))

    grading = grade_item(item_id, response.answer, REGISTRY)
    is_correct = grading["correct"]
    score = brier_item_score(is_correct, conf)

    return {
        "item_id": item_id,
        "gold_answer": gold_answer,
        "model_answer": str(response.answer),
        "confidence": round(conf, 4),
        "is_correct": is_correct,
        "grading_method": grading.get("method", ""),
        "brier_score": round(score, 4),
    }

In [ ]:
# Cell 6 — Main Task: metajudge_calibration

@kbench.task(name="metajudge_calibration")
def metajudge_calibration(llm) -> float:
    """Family A — Confidence Calibration (Monitoring).
    
    Returns anchor-normalized mean 1-Brier score.
    """
    cal_eval = cal_df[["item_id", "question", "gold_answer"]].copy()

    with kbench.client.enable_cache():
        cal_runs = metacog_calibration.evaluate(
            llm=[llm], evaluation_data=cal_eval,
            n_jobs=8, remove_run_files=True,
            stop_condition=lambda runs: len(runs) == len(cal_eval),
            max_attempts=1,
        )

    records = [r.result for r in cal_runs if r.result is not None]
    audit = pd.DataFrame(records)

    raw_score = float(audit["brier_score"].mean())
    normalized = normalize(raw_score, ANCHOR_A_FLOOR, ANCHOR_A_CEIL)

    acc = audit["is_correct"].mean()
    print(f"Calibration: 1-Brier={raw_score:.4f} Acc={acc:.1%} Normalized={normalized:.3f} [{len(audit)} items]")

    # Export audit CSV
    audit.to_csv(os.path.join(OUTPUT_DIR, "calibration_item_audit.csv"), index=False)

    # Export full responses JSON
    import json as _json
    with open(os.path.join(OUTPUT_DIR, "calibration_full_responses.json"), "w") as f:
        _json.dump(records, f, indent=2, default=str)

    return normalized

In [ ]:
# Cell 7 — Run + Submit
metajudge_calibration.run(kbench.llm)
%choose metajudge_calibration

## Methodology

**Axis I — Confidence Calibration (Monitoring).** Per-item: score = 1 − (confidence − outcome)².
Strictly proper scoring rule (Brier 1950). 105 clean items. Correctness graded by deterministic
8-rule engine with tolerance-aware numeric, alias, tri-label, code-output, and fraction grading.
No LLM judge. Normalized to [0, 1] using anchor floor 0.75 (random baseline at 50% confidence)
and ceiling 1.0 (perfect calibration).

Equal weighting across tasks is provably optimal at small n (Davis-Stober 2011; Dawes 1979).
Anchor normalization follows BIG-Bench (Srivastava et al. 2023).